# Ngày 15 · Cây quyết định CART

- Tính độ bẩn Gini và mức giảm Gini của một phép tách
- Tìm phép tách tốt nhất bằng cách quét mọi ngưỡng giữa hai giá trị liền kề
- Dựng cây đệ quy có max_depth và thấy cây sâu quá khớp

**Đọc:**
- ISLR (bản 2), §8.1 The Basics of Decision Trees (8.1.1–8.1.2)
- Hastie, Tibshirani & Friedman, ESL, §9.2 Tree-Based Methods
- Breiman, Friedman, Olshen & Stone (1984), Classification and Regression Trees
- scikit-learn User Guide, §1.10 Decision Trees (mục Mathematical formulation)

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# Breast Cancer Wisconsin (Diagnostic), UCI, có sẵn trong scikit-learn. Cùng phép chia với tuần 2.
_bc = load_breast_cancer()
X_bc, y_bc = _bc.data, 1 - _bc.target   # 1 = ác tính
idx_tr, idx_va = train_test_split(np.arange(len(y_bc)), test_size=0.3, random_state=0, stratify=y_bc)
_scaler = StandardScaler().fit(X_bc[idx_tr])
X_tr, X_va = _scaler.transform(X_bc[idx_tr]), _scaler.transform(X_bc[idx_va])
y_tr, y_va = y_bc[idx_tr], y_bc[idx_va]
from sklearn.tree import DecisionTreeClassifier
_x15 = np.array([[1.], [2.], [3.], [4.], [5.], [6.]]); _y15 = np.array([0, 0, 1, 0, 1, 1])
_sk15 = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_tr, y_tr)
def _best_split_ref(X, y):
    n, d = X.shape; C = int(y.max()) + 1; best = (None, None, 0.0)
    parent = 1 - np.sum((np.bincount(y, minlength=C) / n) ** 2)
    for j in range(d):
        o = np.argsort(X[:, j], kind='stable'); xs, ys = X[o, j], y[o]
        left = np.cumsum(np.eye(C)[ys], axis=0)[:-1]; nl = np.arange(1, n)[:, None]
        right = left[-1] + np.eye(C)[ys[-1]] - left; nr = n - nl
        gain = parent - (nl[:, 0] * (1 - ((left / nl) ** 2).sum(1)) + nr[:, 0] * (1 - ((right / nr) ** 2).sum(1))) / n
        gain = np.where(xs[1:] > xs[:-1], gain, -np.inf)
        i = int(np.argmax(gain))
        if gain[i] > best[2] + 1e-12:
            best = (j, (xs[i] + xs[i + 1]) / 2, float(gain[i]))
    return best
def _same_as_sklearn(build, predict, depth):
    ref = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_tr, y_tr).predict(X_va)
    return np.array_equal(np.asarray(predict(build(X_tr, y_tr, depth), X_va)), ref)


## Gini và phép tách tốt nhất

Gợi ý: Với mỗi cột: sắp xếp, dùng np.cumsum trên one-hot của nhãn để có số đếm bên trái ở mọi vị trí cắt. Chỉ giữ vị trí mà giá trị sau lớn hơn giá trị trước, ngưỡng là trung điểm của hai giá trị đó.

In [ ]:
def gini(y):
    # y: mảng nhãn nguyên 0..C-1. Trả 1 - Σ p_c².
    raise NotImplementedError

def best_split(X, y):
    # Trả (cột, ngưỡng, mức giảm Gini). Không có phép tách nào làm giảm Gini thì trả (None, None, 0.0).
    # Hòa nhau: giữ cột nhỏ hơn, rồi ngưỡng nhỏ hơn.
    raise NotImplementedError


In [ ]:
_report('d15-split',
    _check('Gini đúng cho 2 và 3 lớp', lambda: (np.isclose(gini(np.array([0, 0, 1, 1])), .5) and np.isclose(gini(np.array([1, 1, 1])), 0) and np.isclose(gini(np.array([0, 1, 2])), 2 / 3), 'gini([0,0,1,1]) = 0,5; gini([0,1,2]) = 2/3.')),
    _check('Ví dụ tính tay: (cột 0, t = 2,5, giảm 0,25)', lambda: ((lambda b: b[0] == 0 and np.isclose(b[1], 2.5) and np.isclose(b[2], .25))(best_split(_x15, _y15)), 'Hòa với t = 4,5 thì giữ ngưỡng nhỏ hơn.')),
    _check('Không cắt giữa hai giá trị bằng nhau', lambda: ((lambda b: b[0] == 0 and np.isclose(b[1], 1.5))(best_split(np.array([[1.], [1.], [1.], [2.]]), np.array([0, 0, 1, 1]))), 'x = [1, 1, 1, 2]: chỉ có ngưỡng 1,5.')),
    _check('Gốc cây khớp DecisionTreeClassifier trên Breast Cancer', lambda: ((lambda b: b[0] == _sk15.tree_.feature[0] and np.isclose(b[1], _sk15.tree_.threshold[0], atol=1e-5))(best_split(X_tr, y_tr)), f'scikit-learn chọn cột {_sk15.tree_.feature[0]} ({_bc.feature_names[_sk15.tree_.feature[0]]}).')),
)


## Dựng cây đệ quy và dự đoán

Gợi ý: Nút là dict. Dừng khi depth = max_depth, khi số mẫu < min_samples_split, khi nút đã thuần, hoặc khi best_split trả None. Lá dự đoán lớp đa số (hòa thì lớp nhỏ hơn: np.argmax trên np.bincount).

In [ ]:
def build_tree(X, y, max_depth, min_samples_split=2, depth=0):
    # Lá: {'leaf': True, 'value': lớp}. Nút trong: {'leaf': False, 'feature', 'threshold', 'left', 'right'}.
    # Mẫu có X[:, feature] <= threshold đi sang trái. Dùng best_split của bài trên.
    raise NotImplementedError

def predict_tree(tree, X):
    raise NotImplementedError


In [ ]:
best_split = globals().get('best_split') or _best_split_ref
_report('d15-tree',
    _check('Khớp scikit-learn với max_depth = 1, 2, 3', lambda: (all(_same_as_sklearn(build_tree, predict_tree, d) for d in (1, 2, 3)), 'So dự đoán trên 171 mẫu validation.')),
    _check('Cây không giới hạn thuộc lòng tập train', lambda: (np.mean(predict_tree(build_tree(X_tr, y_tr, 100), X_tr) == y_tr) == 1.0, 'Accuracy train = 1 là dấu hiệu quá khớp, không phải mục tiêu.')),
    _check('max_depth = 0 cho một lá dự đoán lớp đa số', lambda: (np.all(predict_tree(build_tree(X_tr, y_tr, 0), X_va) == 0), 'Lớp đa số của train là 0 (lành tính).')),
)


## Dự án

Trên Titanic, thay LogisticRegression trong pipeline ngày 14 bằng DecisionTreeClassifier(max_depth=d, random_state=0) với d = 1…10. Vẽ accuracy train và accuracy stratified 5-fold theo d, chọn d và giải thích chỗ hai đường tách xa nhau. In ba tầng đầu của cây bằng sklearn.tree.export_text.